# 4. EDA categórico y cruces

Este cuaderno describe la distribución de las variables categóricas del dataset procesado y sus asociaciones con la recompra.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'src').exists():
    ROOT_DIR = ROOT_DIR.parent
sys.path.insert(0, str(ROOT_DIR / 'src'))

from analisis import (
    crear_rango_offervalue,
    matriz_correlacion_spearman,
    resumen_correlaciones_repeater,
    tabla_frecuencia,
    tasa_recompra_por_grupo,
)
from config import FIGURES_DIR, TABLES_DIR
from preprocesamiento import cargar_dataset_modelo

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## Carga y validación

In [ ]:
dataset = cargar_dataset_modelo()
columnas_categoricas = ['chain', 'market', 'category', 'company', 'brand']
columnas_requeridas = [*columnas_categoricas, 'offervalue', 'repeater']
faltantes_columnas = set(columnas_requeridas).difference(dataset.columns)
assert not faltantes_columnas, f'Columnas ausentes: {sorted(faltantes_columnas)}'

print(f'Dataset cargado: {dataset.shape[0]:,} filas y {dataset.shape[1]} columnas.')
print('La variable objetivo repeater está disponible.')

faltantes_relevantes = dataset[columnas_requeridas + ['dias_ultima_compra_categoria']].isna().sum()
validacion = pd.DataFrame({
    'verificacion': ['filas', 'columnas', 'repeater', *columnas_categoricas, 'offervalue'],
    'resultado': [dataset.shape[0], dataset.shape[1], *(col in dataset.columns for col in ['repeater', *columnas_categoricas, 'offervalue'])],
})
display(validacion)
display(faltantes_relevantes.rename('faltantes').to_frame())

## Distribución de variables categóricas

Los identificadores son anónimos; por ello se presentan por frecuencia sin atribuirles significado comercial.

In [ ]:
tablas_frecuencia = {}
for variable in columnas_categoricas:
    tabla = tabla_frecuencia(dataset, variable)
    tablas_frecuencia[variable] = tabla
    tabla.to_csv(TABLES_DIR / f'04_frecuencia_{variable}.csv', index=False)

    top = tabla.head(15)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(top['grupo'].astype(str), top['frecuencia'], color='#4472C4')
    ax.set_title(f'15 categorías más frecuentes de {variable}')
    ax.set_xlabel(variable)
    ax.set_ylabel('Número de clientes')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'04_frecuencia_{variable}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f'Frecuencia de {variable}')
    display(tabla.head())

## Tasas de recompra por grupo

In [ ]:
def graficar_tasa(tabla, titulo, ruta, limitar_a_15=False):
    """Guarda un gráfico de tasa de recompra, con grupos ordenados por tamaño."""
    grafico = tabla.nlargest(15, 'n') if limitar_a_15 else tabla
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(grafico['grupo'].astype(str), grafico['tasa_recompra'], color='#70AD47')
    ax.set_title(titulo)
    ax.set_xlabel('Grupo')
    ax.set_ylabel('Tasa de recompra')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.close(fig)

tasa_chain = tasa_recompra_por_grupo(dataset, 'chain')
tasa_chain.to_csv(TABLES_DIR / '04_tasa_recompra_chain.csv', index=False)
graficar_tasa(
    tasa_chain,
    'Tasa de recompra por cadena (15 grupos con más observaciones)',
    FIGURES_DIR / '04_tasa_recompra_chain.png',
    limitar_a_15=True,
)
display(tasa_chain.head())

tasa_category = tasa_recompra_por_grupo(dataset, 'category')
tasa_category.to_csv(TABLES_DIR / '04_tasa_recompra_category.csv', index=False)
graficar_tasa(
    tasa_category,
    'Tasa de recompra por categoría (15 grupos con más observaciones)',
    FIGURES_DIR / '04_tasa_recompra_category.png',
    limitar_a_15=True,
)
display(tasa_category.head())

dataset_con_rangos = dataset.copy()
dataset_con_rangos['rango_offervalue'] = crear_rango_offervalue(dataset_con_rangos['offervalue'])
tasa_offervalue = tasa_recompra_por_grupo(dataset_con_rangos, 'rango_offervalue')
orden_rangos = ['<= 1.00', '1.00–1.50', '1.50–2.00', '> 2.00']
tasa_offervalue['grupo'] = pd.Categorical(tasa_offervalue['grupo'], categories=orden_rangos, ordered=True)
tasa_offervalue = tasa_offervalue.sort_values('grupo').reset_index(drop=True)
tasa_offervalue.to_csv(TABLES_DIR / '04_tasa_recompra_offervalue.csv', index=False)
graficar_tasa(
    tasa_offervalue,
    'Tasa de recompra por rango de valor de oferta',
    FIGURES_DIR / '04_tasa_recompra_offervalue.png',
)
display(tasa_offervalue)

## Correlaciones con la variable objetivo

Se usan correlaciones de Spearman y solo variables numéricas de comportamiento. Se excluyen identificadores, atributos categóricos de la oferta y `repeattrips`, que deriva directamente de la variable objetivo.

In [ ]:
excluidas_correlacion = {
    'id', 'offer', 'chain', 'market', 'category', 'company', 'brand',
    'offervalue', 'quantity', 'repeattrips', 'repeater',
}
variables_comportamiento = [
    columna for columna in dataset.select_dtypes(include=[np.number]).columns
    if columna not in excluidas_correlacion
]

matriz_correlacion = matriz_correlacion_spearman(dataset, variables_comportamiento)
matriz_correlacion.to_csv(TABLES_DIR / '04_matriz_correlacion.csv')
correlaciones_repeater = resumen_correlaciones_repeater(matriz_correlacion)
correlaciones_repeater.to_csv(TABLES_DIR / '04_correlaciones_repeater.csv', index=False)
display(correlaciones_repeater.head(10))

variables_figura = correlaciones_repeater['variable'].head(15).tolist()
orden_figura = [*variables_figura, 'repeater']
matriz_figura = matriz_correlacion.loc[orden_figura, orden_figura]
fig, ax = plt.subplots(figsize=(11, 9))
imagen = ax.imshow(matriz_figura, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(orden_figura)), orden_figura, rotation=60, ha='right')
ax.set_yticks(range(len(orden_figura)), orden_figura)
for i in range(len(orden_figura)):
    for j in range(len(orden_figura)):
        ax.text(j, i, f'{matriz_figura.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(imagen, ax=ax, label='Correlación de Spearman')
ax.set_title('Correlaciones de las variables de comportamiento más asociadas con repeater')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '04_matriz_correlacion.png', dpi=150, bbox_inches='tight')
plt.close(fig)

## Faltantes y valores extremos

In [ ]:
faltantes = dataset.isna().sum()
faltantes = faltantes[faltantes.gt(0)].sort_values(ascending=False).rename('faltantes').to_frame()
display(faltantes)

variables_extremos = [
    'n_lineas', 'n_visitas', 'gasto_total', 'ticket_promedio',
    'n_compras_categoria', 'n_compras_producto',
]
resumen_extremos = pd.DataFrame({
    'mediana': dataset[variables_extremos].median(),
    'percentil_99': dataset[variables_extremos].quantile(0.99),
    'maximo': dataset[variables_extremos].max(),
})
display(resumen_extremos)

print(
    'Los faltantes se conservan: dias_ultima_compra_categoria identifica a quienes '
    'nunca compraron antes la categoría ofertada y la variable '
    'nunca_compro_categoria mantiene esa información.'
)
print(
    'No se eliminan valores extremos en esta etapa: el EDA cuantitativo previo mostró '
    'colas largas y retirar filas podría alterar las tasas de recompra por grupo.'
)